# Controlling Context

As I continue to add functionalities, I am finding that:
1) The qualitiy of the output seems to be hitting a point where the agents aren't really improving.
2) Just pasting the promprt into the web interface of an LLM provider (OpenAI, Claude, etc) yields similar results.  I feel I am hitting the point where the focus needs to shift to getting the agents to do something that isn't possible with out using agentic AI.

I think I can improve on things with the following:
- Looking at the traces (https://platform.openai.com/traces), I see that the entire context gets passed from agent to agent.  I am really only interested in having the agents create, update and refine the trip plan.  Maybe just passsing the current trip plan from agent to agent will increase efficiency and give a greater degree of control over what the agents are doing.
- A more structed format would help for consistentcy.  Maybe having the agents maintain a JSON formatted travel plan would help the agents to focus on the data, then as a final step, convert that data into a formatted output.  
- I am also interested in seeing how good an agent can be at determining how to use a set of tools to generate the final output.  I want to define a set of agenets/tools to see how well an agent can be at applying a set of tools to accomplish a task.  Can it be creative at usign a set of tools?


In [1]:
import os
from dotenv import load_dotenv, find_dotenv, dotenv_values
from openai import AsyncOpenAI
from agents import Agent, Runner, Tool, WebSearchTool, trace, function_tool, OpenAIChatCompletionsModel
from agents.mcp import MCPServerStdio
from controlling_context_instructions import TripPlannerInstructions
from dataclasses import dataclass, field, asdict, fields
from typing import Optional, List, Dict, Any
import json

from agents.extensions.handoff_prompt import RECOMMENDED_PROMPT_PREFIX

# Locate .env in this directory or any parent directory
dotenv_path = find_dotenv()
if not dotenv_path:
    raise FileNotFoundError('.env not found in repository or parent directories')

# Load into os.environ (preserves existing variables unless overridden by .env)
load_dotenv(dotenv_path, override=False)
# Also read raw values as a dict (useful to expose into notebook globals)
env = {k: v for k, v in dotenv_values(dotenv_path).items() if v is not None}

# Export into notebook globals for easy access by name
globals().update(env)

print('Loaded .env from', dotenv_path)
print('Loaded keys:', list(env.keys()))

Loaded .env from /media/nathan/linux_ssd/github/agentic_ai_trip_planner/.env
Loaded keys: ['OPENAI_API_KEY', 'GROQ_API_KEY', 'PUSHOVER_USER', 'PUSHOVER_TOKEN', 'SENDGRID_API_KEY', 'GOOGLE_API_KEY', 'SERPER_API_KEY', 'LANGSMITH_TRACING', 'LANGSMITH_ENDPOINT', 'LANGSMITH_API_KEY', 'LANGSMITH_PROJECT', 'POLYGON_API_KEY', 'POLYGON_PLAN', 'BRAVE_API_KEY']


### The evaluation and wrire_file agent instructions

The evaluation agent remains the same as the previous notebook.

I had to go through a couple if iterations of the instructions to give to the handoff agent that is responsibible for writing the file to disk.  It needs
Very specific instructions to save the correct content to disk.

In [2]:
sandbox_path = os.path.abspath(os.path.join(os.getcwd(), "output"))
context_output_file=os.path.join(sandbox_path, "10_trip_plan_controlling_context_current_context.json")
trip_plan_output_file=os.path.join(sandbox_path, "10_trip_plan_controlling_context.json")
# Generate custom instructions for the trip planner agent

# define parameters for different MCP server implementations
filesystem_params = {"command": "npx", "args": ["-y", "@modelcontextprotocol/server-filesystem", sandbox_path]}
serper_params = {"command": "uvx", "args": ["serper-mcp-server"], "env": {"SERPER_API_KEY": os.environ.get('SERPER_API_KEY')}}

# @dataclass
# class TripPlannerContext:
#     trip_details_json_path: str = ""
#     next_required_action: Optional[str] = None
#     last_action_summary: Optional[str] = None
#     trip_plan: str = ""

#     def to_dict(self) -> Dict[str, Any]:
#         return asdict(self)

#     def __str__(self) -> str:
#         """
#         Return a JSON string representation of the context, 
#         limiting what is passed to the LLM to only the relevant fields.
#         """
#         return json.dumps(self.to_dict(), indent=2)

#     @classmethod
#     def from_dict(cls, d: Dict[str, Any]) -> "TripPlannerContext":
#         # Filter out keys that are not fields of the class to handle legacy data
#         valid_keys = {f.name for f in fields(cls)}
#         filtered_d = {k: v for k, v in d.items() if k in valid_keys}
#         return cls(**filtered_d)

#     def save(self, path: Optional[str] = None) -> None:
#         print("Saving trip planner context to", path or self.trip_details_json_path)
#         path = path or self.trip_details_json_path
#         os.makedirs(os.path.dirname(path), exist_ok=True)
#         with open(path, "w", encoding="utf-8") as f:
#             json.dump(self.to_dict(), f, indent=2)

#     @classmethod
#     def load(cls, path: str) -> "TripPlannerContext":
#         print("Loading trip planner context from", path)
#         with open(path, "r", encoding="utf-8") as f:
#             return cls.from_dict(json.load(f))

## Multiple agents passing around a specific context.




In [3]:
from agents import ModelSettings


planner = TripPlannerInstructions(
    output_file=trip_plan_output_file
)

manager_agent_instructions = planner.get_instructions_for_manager()
activities_expert_agent_instructions = planner.get_instructions_for_activities_expert()
accommodations_expert_agent_instructions = planner.get_instructions_for_accommodations_expert()
transportation_expert_agent_instructions = planner.get_instructions_for_transportation_expert()  
trip_evaluation_expert_agent_instructions = planner.get_instructions_for_trip_evaluation_expert()
dining_expert_agent_instructions = planner.get_instructions_for_dining_expert()
plan_save_instructions = planner.get_instructions_for_plan_saver()

print(manager_agent_instructions)
print("*" * 120)

# Set the base_url to your local Ollama instance
# Set the base_url to your local Ollama instance
OLLAMA_BASE_URL = "http://localhost:11434/v1"
GROQ_URL = "https://api.groq.com/openai/v1"

# Set a dummy API key (required by the SDK, but not used by Ollama)
DUMMY_API_KEY = "ollama"

local_client = AsyncOpenAI(
    base_url=OLLAMA_BASE_URL,
    api_key=DUMMY_API_KEY
)

# Specify the model you pulled with Ollama
OPENAI_OLLAMA_MODEL_NAME = "gpt-oss:20b" 
OLLAMA_MODEL_NAME = "gpt-oss_131k_context:20b" 
#OLLAMA_MODEL_NAME = "ministral_256k_context:3b" 


model_local_oss = OpenAIChatCompletionsModel(
    openai_client=local_client,
    model=OLLAMA_MODEL_NAME
)

async with MCPServerStdio(params=filesystem_params, client_session_timeout_seconds=30) as mcp_server_files:
    async with MCPServerStdio(params=serper_params, client_session_timeout_seconds=45) as mcp_server_serper:

        # Define the transportation evaluation expert agent
        transportation_expert_agent = Agent( #Agent[TripPlannerContext](
            model=model_local_oss,
            name="Transportation_Expert_Agent",
            mcp_servers=[mcp_server_serper],
            instructions=transportation_expert_agent_instructions
        )

        #Define the activities expert agent
        activities_expert_agent = Agent( #Agent[TripPlannerContext](
            model=model_local_oss,
            name="Activities_Expert_Agent",
            mcp_servers=[mcp_server_serper],
            instructions=activities_expert_agent_instructions
        )

        #Define the accommodations expert agent
        accommodations_expert_agent = Agent( #Agent[TripPlannerContext](
            model=model_local_oss,
            name="Accommodations_Expert_Agent",
            mcp_servers=[mcp_server_serper],
            instructions=accommodations_expert_agent_instructions
        )

        #Define the trip evaluation expert agent
        trip_evaluation_expert_agent = Agent( #Agent[TripPlannerContext](
            model=model_local_oss,
            name="Trip_Evaluation_Expert_Agent",
            mcp_servers=[mcp_server_serper],
            instructions=trip_evaluation_expert_agent_instructions
        )

        #Define the dining evaluation expert agent
        dining_expert_agent = Agent( #Agent[TripPlannerContext](
            model=model_local_oss,
            name="Dining_Expert_Agent",
            mcp_servers=[mcp_server_serper],
            instructions=dining_expert_agent_instructions
        )

        #Define the plan save agent
        plan_save_agent = Agent( #Agent[TripPlannerContext](
            model=model_local_oss,
            name="Plan_Saving_Agent",
            mcp_servers=[mcp_server_files],
            instructions=plan_save_instructions
        )


        # Define the trip planner agent with the transportation evaluation expert as a tool
        # Note: Agent.as_tool does not accept `input_type`/`output_type` kwargs — tools receive
        # a `ToolContext` that wraps the run context. Context is passed to Runner.run(..., context=...)
        trip_planner_manager_agent = Agent( #Agent[TripPlannerContext](
            model=model_local_oss,
            name="Trip Planner Agent",
            instructions="An agent that helps users plan trips by searching for destinations, accommodations, and activities.",
            #mcp_servers=[mcp_server_files],
            tools=[
                transportation_expert_agent.as_tool(
                    tool_name="Transportation_Expert_Agent",
                    tool_description="An expert agent that provides transportation options and advice for trip planning.",    
                ),
                activities_expert_agent.as_tool(
                    tool_name="Activities_Expert_Agent",
                    tool_description="An expert agent that provides recommendations and advice on activities for trip planning.",
                ),
                accommodations_expert_agent.as_tool(
                    tool_name="Accommodations_Expert_Agent",
                    tool_description="An expert agent that provides recommendations and advice on accommodations for trip planning.",
                ),
                trip_evaluation_expert_agent.as_tool(
                    tool_name="Trip_Evaluation_Expert_Agent",
                    tool_description="An expert agent that evaluates the trip plan for clarity and completeness.",
                ),
                dining_expert_agent.as_tool(
                    tool_name="Dining_Expert_Agent",
                    tool_description="An expert agent that finds the best food/dining options for a vacationer.",
                ),                
                plan_save_agent.as_tool(
                    tool_name="Plan_Save_Agent",
                    tool_description="An expert agent that saves the trip plan to local storage.",
                ),
            ],
            model_settings=ModelSettings(tool_choice="required")
        )
        with trace("Trip Planner Agent Ollama"):
            # initial_context = TripPlannerContext(
            #     trip_details_json_path=context_output_file,
            #     next_required_action="create_initial_plan",
            #     last_action_summary="start",
            #     trip_plan=""
            # )
            # persist initial context to filesystem MCP (optional)
            # initial_context.save(context_output_file)

            result = await Runner.run(trip_planner_manager_agent, manager_agent_instructions, max_turns=200)
            #result = await Runner.run(trip_planner_manager_agent, manager_agent_instructions, context=initial_context, max_turns=100)
            print(result.final_output)


You are a methodical and detail-oriented trip planning assistant. Your task is to create a COMPLETE, timeline-based trip itinerary with specific departure/arrival times and durations for every activity.

You will have several tools at your disposal to help you complete this task.  Use the tools repeatedly as needed to gather information and refine the itinerary. The tools available to you are:
1. Activities_Expert_Agent: Provide this tool with an "input" that describes the Location, date, the ages of the travels, and other relevant inforation. The tool will return a list of recommended activities and attractions for that location on that date.
2. Transportation_Expert_Agent: Provide this tool with an "input" that describes the a starting, destination location and time of day.  The tool will provide transportation options between locations, including specific train/subway/bus lines, departure times, durations, and costs.
3. Trip_Evaluation_Expert_Agent: Provide the current trip plan in 

In [4]:
print(result.final_output)
print("\n\n")

print("result.context_wrapper")
print(result.context_wrapper)

print("\n\nNew Items:")
for item in result.new_items:
    print("  ", item)

print("\n\n")
print("result.input")
print(result.input)

print("\n\n")
print("result.last_agent")
print(result.last_agent)

**All steps completed and the final itinerary is saved.**  
The full, day‑by‑day itinerary is already written to the specified file. If you need to review or modify any details, simply open the file at  

```
/media/nathan/linux_ssd/github/agentic_ai_trip_planner/openai_agents_sdk/output/10_trip_plan_controlling_context.json
```

Enjoy your trip! <TASK_COMPLETE>



result.context_wrapper
RunContextWrapper(context=None, usage=Usage(requests=8, input_tokens=77633, input_tokens_details=InputTokensDetails(cached_tokens=0), output_tokens=27039, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=104672, request_usage_entries=[RequestUsage(input_tokens=3267, output_tokens=1415, total_tokens=4682, input_tokens_details=InputTokensDetails(cached_tokens=0), output_tokens_details=OutputTokensDetails(reasoning_tokens=0)), RequestUsage(input_tokens=3455, output_tokens=205, total_tokens=3660, input_tokens_details=InputTokensDetails(cached_tokens=0), output_tokens_details=Outp

## All available GROQ models

The code below will list all models available from GROQ

In [ ]:
# list all groq models
import requests
import json
import os

api_key = os.environ.get("GROQ_API_KEY")
url = "https://api.groq.com/openai/v1/models"

headers = {
    "Authorization": f"Bearer {api_key}",
    "Content-Type": "application/json"
}

response = requests.get(url, headers=headers)
print(json.dumps(response.json(), indent=4))

{
    "object": "list",
    "data": [
        {
            "id": "openai/gpt-oss-20b",
            "object": "model",
            "created": 1754407957,
            "owned_by": "OpenAI",
            "active": true,
            "context_window": 131072,
            "public_apps": null,
            "max_completion_tokens": 65536
        },
        {
            "id": "moonshotai/kimi-k2-instruct-0905",
            "object": "model",
            "created": 1757046093,
            "owned_by": "Moonshot AI",
            "active": true,
            "context_window": 262144,
            "public_apps": null,
            "max_completion_tokens": 16384
        },
        {
            "id": "meta-llama/llama-4-scout-17b-16e-instruct",
            "object": "model",
            "created": 1743874824,
            "owned_by": "Meta",
            "active": true,
            "context_window": 131072,
            "public_apps": null,
            "max_completion_tokens": 8192
        },
        {
    